## working with agents on top of carteirinha extracted database

### this also should incorporate the base workflow style:
input: blob, id -> image, id -> llm ->  output: convenio, plano, nome da pessoa e número da carteirinha


In [37]:
import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
#strands agetinc workflow
from strands.models import BedrockModel
from strands import Agent

#dont limit the visualization of all the colluns of a pandas dataframe
pd.set_option("display.max_columns", None)

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## credentials config

In [38]:
@dataclass
class AppConstants:
    BEDROCK_DEFAULT_MODEL_ID: str = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    DEFAULT_PROMPTS_DIR: str = "prompts/"
    S3_BUCKET_NAME: str = "agente-ai-carteirinha"
    S3_RESULTS_PREFIX: str = "resultados"
    S3_DEBUG_PREFIX: str = "debug"
    STREAMING: bool = False
    CACHE_PROMPT = "default"
    RETRIES: Dict[str, int] = field(default_factory=lambda: {"max_attempts": 3, "mode": "standard"})
    CONNECTION_TIMEOUT: int = 5
    READ_TIMEOUT: int = 60
    TEMPERATURE: float = 0.05
    TOP_P: float = 0.95

In [39]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    AWS_SERVICE_NAME: str
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

In [40]:
def create_boto3_session(
    settings: Settings, config: Optional[Config] = None
) -> boto3.Session:
    try:
        logger.info(
            f"Criando sessão boto3 para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        session = boto3.Session(
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
        )
        logger.info("Sessão boto3 criada com sucesso.")
        return session
    except Exception as e:
        logger.critical(f"Não foi possível criar a sessão boto3: {e}")
        raise

## geting the service ready to use
### model configuration

In [41]:

app_constants = AppConstants()
settings = Settings()

# Create a custom boto3 session

session = create_boto3_session(settings)

# Create a Bedrock model with the custom session
bedrock_model = BedrockModel(
    model_id=settings.BEDROCK_MODEL_ID,
    boto_session=session,
    streaming=False,
    temperature=app_constants.TEMPERATURE,
    top_p=app_constants.TOP_P,
    boto_client_config=Config(
        retries=app_constants.RETRIES,
        connect_timeout=app_constants.CONNECTION_TIMEOUT,
        read_timeout=app_constants.READ_TIMEOUT
    )
)

{"timestamp": "2025-08-12T07:41:00", "level": "INFO", "name": "__main__", "message": "Criando sessão boto3 para a região: us-east-1...", "filename": "244079052.py", "lineno": 5}
{"timestamp": "2025-08-12T07:41:00", "level": "INFO", "name": "__main__", "message": "Sessão boto3 criada com sucesso.", "filename": "244079052.py", "lineno": 13}


In [42]:
from strands import Agent

agent = Agent(model=bedrock_model)

prompt = "Tell me about Amazon Bedrock."
response = agent(prompt=prompt)


# Amazon Bedrock

Amazon Bedrock is a fully managed service that provides access to a range of foundation models (FMs) through a unified API. It allows developers to build and scale generative AI applications without having to manage the underlying infrastructure.

Key features include:

- **Model variety**: Access to leading FMs from AI companies like Anthropic, AI21 Labs, Cohere, Meta, Stability AI, and Amazon's own models
- **Customization options**: Fine-tune models with your own data
- **Security and privacy**: Enterprise-grade security with private access, encryption, and data protection
- **Integration**: Seamlessly connects with other AWS services
- **Serverless experience**: No infrastructure management required

Bedrock enables businesses to build various AI applications including content generation, summarization, classification, Q&A systems, and chatbots while maintaining control over their data.

In [75]:
# system prompt will be passed as a txt file
class AgentCarteirinha():
    """ This agent will use a multimodal LLM.
    The user will input a list of images on base64, and the agent will process them accordingly, based
    in its agent definition by the system prompt and the passed images.
    The output should be a well structured json"""
    def __init__(self, model: BedrockModel, system_prompt: str, data_model: PydanticBaseModel):
        self.model = model
        self.system_prompt = system_prompt
        self.agent = Agent(model=self.model, system_prompt=system_prompt)
        self.data_model = data_model

    def extract_text_from_pdf(self, pdf_bytes: bytes) -> Dict:
        """ This function will receive a pdf file in bytes format,
        and will return a structured json with the extracted data.
        We append the pdf file and our agent definition as prompt to send
        on a single request."""
        pdf_content =  {
            "document": {
                "format": "pdf",
                "name": f"pdf_blob_{len(pdf_bytes)}.pdf",
                "source": {
                    "bytes": pdf_bytes
                }
            }
        }

        messages = [
            {"role": "user",
             "content": pdf_content
            }
        ]
        prompt="Extract the data from the carteirinha PDF and return it in a structured JSON format. follow the system prompt"
        return self.agent(messages=messages, prompt=prompt)



    def extract_data_from_images_bytes(self, list_images_bytes: List[bytes]) -> Dict:
        """ This function will receive a list of images in bytes png format,
        and will return a structured json with the extracted data.
        We append max of 20 images and our agent definition as prompt to send
        on a single request."""

        images_text_list = []
        for idx, img_bytes in enumerate(list_images_bytes[:20], start=0):
            images_text_list.append({
                "document": {
                    "format": "png",
                    "name": f"image_{idx}",
                            "source": {
                                "bytes": img_bytes
                            }
                        }
                    })

        messages = [
            {"role": "user",
             "content": images_text_list
            }
        ]

        prompt="Extract the data from the png documents provided and return it in a structured JSON format. follow the system prompt"

        return self.agent(messages=messages, prompt=prompt)
        # return self.agent.structured_output(output_model=self.data_model, prompt=images_text_list)



In [44]:
# image utils
MAX_IMAGES_PER_BLOB = 20  # Maximum number of images per blob
def converter_blob_para_imagens(blob: bytes, extensao: str) -> List[Image.Image]:
    imagens = []
    ext = extensao.lower().strip(".") if extensao else ""
    try:
        if ext == "pdf":
            with fitz.open(stream=blob, filetype="pdf") as pdf_doc:
                logger.info(f"Processando PDF com {len(pdf_doc)} página(s)...")
                for pagina in pdf_doc: # limit to 20 pages
                    if len(imagens) >= MAX_IMAGES_PER_BLOB:
                        logger.warning(f"⚠️ BLOB has reached the maximum limit of {MAX_IMAGES_PER_BLOB} images.")
                        break
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(3.0, 3.0), alpha=False)
                    imagens.append(Image.open(io.BytesIO(pix.tobytes("png"))))
        elif ext in ["jpg", "jpeg", "png", "bmp"]:
            imagens.append(Image.open(io.BytesIO(blob)))
        else:
            logger.warning(f"Formato de arquivo não suportado: '{ext}'.")
    except Exception as e:
        logger.error(f"Erro ao converter BLOB para imagem (ext: .{ext}): {e}")
    return imagens


def aplicar_clahe(imagem: Image.Image) -> Image.Image:
    try:
        imagem_cv = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
        imagem_cinza = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(imagem_cinza))
    except Exception:
        return imagem

def imagens_para_bytes(lista_imagens: List[Image.Image], formato: str = "PNG") -> List[bytes]:
    imagens_bytes = []
    TARGET_BYTES = 3.5 * 1024 * 1024
    for img in lista_imagens:
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        for quality in range(95, 15, -10):
            buffer = io.BytesIO()
            img.save(buffer, format="png", quality=quality)
        imagens_bytes.append(buffer.getvalue())
    return imagens_bytes

## load the dataset and explore for prompt ideas

In [45]:
import sqlalchemy
data_path = "gold_carteirinha_database.sqlite"
engine = sqlalchemy.create_engine(f"sqlite:///{data_path}")
df_merged_final = pd.read_sql("SELECT * FROM carteirinha", engine)

df_merged_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 207 entries, 0 to 206
Data columns (total 37 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   CD_AVISO_CIRURGIA             207 non-null    int64 
 1   CD_DOCUMENTO_ANEXO_CIRURGICO  207 non-null    int64 
 2   CD_GUIA                       207 non-null    int64 
 3   TP_GUIA                       207 non-null    object
 4   TP_SITUACAO                   207 non-null    object
 5   LO_DOCUMENTO_ANEXO_CIRURGICO  207 non-null    object
 6   DS_EXTENSAO                   207 non-null    object
 7   DT_ANEXO                      207 non-null    object
 8   DS_DOCUMENTO_ANEXO            207 non-null    object
 9   CD_PACIENTE                   207 non-null    int64 
 10  NR_CARTEIRA                   207 non-null    object
 11  CD_CONVENIO                   207 non-null    int64 
 12  NM_EMPRESA                    0 non-null      object
 13  DT_INTEGRA          

In [46]:
# get a dataframe that groups by NOME_CONVENIO
df_grouped = df_merged_final[["NOME_CONVENIO", "NR_CARTEIRA"]].groupby("NOME_CONVENIO").agg(list).reset_index()


# geting a new df  with only the first ocurrency of NR_CARTEIRA FOR EACH NOME_CONVENIO
df_first_occurrence = df_grouped.explode("NR_CARTEIRA").drop_duplicates(subset=["NOME_CONVENIO"], keep="first").reset_index(drop=True)

# NOW lets ADD the len of NR_CARTEIRA col
df_first_occurrence["NR_CARTEIRA_LEN"] = df_first_occurrence["NR_CARTEIRA"].apply(lambda x: len(x) if isinstance(x, str) else 0)

df_first_occurrence = df_first_occurrence.sort_values(by="NR_CARTEIRA_LEN", ascending=False).reset_index(drop=True)

print(df_first_occurrence)

                    NOME_CONVENIO        NR_CARTEIRA  NR_CARTEIRA_LEN
0           SUL AMERICA DIRETO BH  88888482109270016               17
1             STELLANTIS SAUDE MG  00010001091710017               17
2                     SUL AMERICA  88888483405330026               17
3                           CASSI   0300068161720067               16
4         CAIXA ECONOMICA FEDERAL   0101407360000254               16
5                    BLUE COMPANY   0000000620569300               16
6         POSTAL SAUDE - CORREIOS   0184073136030275               16
7                            IPSM   6500116540743112               16
8                  UNIMED SEGUROS   9941868288890028               16
9                        BRADESCO    891722600159007               15
10             BRADESCO OPERADORA    954420014320014               15
11             PLAN ASSISTE - MPF     10511003380000               14
12                      CARE PLUS       090900038401               12
13              PETR

### load a blob to our agent and see the response

In [47]:
# data model
class CarteirinhaExtraida(PydanticBaseModel):
    """Define a estrutura dos dados extraídos da carteirinha."""
    cd_aviso_cirurgia: Optional[str] = Field(None, description="ID do aviso de cirurgia")
    confidence_its_carteirinha: Optional[float] = Field(None, description="Confiança na extração da carteirinha.")
    real_carteirinha_num: Optional[str] = Field(None, description="Número real da carteirinha.")
    convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
    plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
    nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
    numero_carteirinha: Optional[str] = Field(None, description="O número de identificação da carteirinha.")

In [48]:
# get the entire row from example blob
df_merged_final.iloc[2]

CD_AVISO_CIRURGIA                                                          792061
CD_DOCUMENTO_ANEXO_CIRURGICO                                              2469151
CD_GUIA                                                                  18191254
TP_GUIA                                                                         I
TP_SITUACAO                                                                     A
LO_DOCUMENTO_ANEXO_CIRURGICO    b'%PDF-1.4\n1 0 obj\n<<\n/Title (\xfe\xff\x00E...
DS_EXTENSAO                                                                  .pdf
DT_ANEXO                                               2025-01-08 15:48:29.000000
DS_DOCUMENTO_ANEXO                                           Carteira do convênio
CD_PACIENTE                                                                925516
NR_CARTEIRA                                                     88888483405330026
CD_CONVENIO                                                                   110
NM_EMPRESA      

In [76]:
example_blob = df_merged_final.iloc[2]["LO_DOCUMENTO_ANEXO_CIRURGICO"]

images = converter_blob_para_imagens(example_blob, "pdf")
# optional: save the image 
save_path = "docs/images/"
os.makedirs(save_path, exist_ok=True)
for i, img in enumerate(images):
    img.save(f"{save_path}/image_{i}.png")

images_with_clahe = [aplicar_clahe(img) for img in images]
images_bytes_list = imagens_para_bytes(images_with_clahe)



#transform the images to png format
images_png = [img.convert("RGBA") for img in images_with_clahe]



{"timestamp": "2025-08-12T07:49:11", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "708635088.py", "lineno": 9}


In [50]:
# load the query from the docs/carteirinha_agent_prompt_v3.txt
with open("docs/carteirinha_agent_prompt_v4.txt", "r", encoding="utf-8") as file:
    carteirinha_agent_prompt = file.read()


In [51]:
carteirinha_agent_prompt = carteirinha_agent_prompt.strip()


In [52]:
# same prompt but here

prompts = {
    "prompt_extracao_carteirinha_v1": """
        A imagem fornecida é uma carteirinha de convênio de saúde. Analise a imagem e extraia as seguintes informações em formato JSON:
        - convenio: O nome da operadora do plano de saúde.
        - plano: O tipo ou nome do plano (ex: "Plano Prata", "Enfermaria").
        - nome_pessoa: O nome completo do beneficiário.
        - numero_carteirinha: O número de identificação ou matrícula da carteirinha.

        Se alguma informação não for encontrada, retorne `null` para o campo correspondente.
        O JSON deve ter a seguinte estrutura:
        {
        "convenio": "string",
        "plano": "string",
        "nome_pessoa": "string",
        "numero_carteirinha": "string"
        }
        """,

        "prompt_extracao_carteirinha_v2": """
        Você receberá uma ou mais imagens. Analise todas e verifique se alguma delas contém uma carteirinha de convênio de saúde.
        Uma carteirinha normalmente possui estes campos:
        - convenio: O nome da operadora do plano de saúde.
        - plano: O tipo ou nome do plano (ex: "Plano Prata", "Enfermaria").
        - nome_pessoa: O nome completo do beneficiário.
        - numero_carteirinha: O número de identificação ou matrícula da carteirinha.
        onde, de forma mais tecnica:
            convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
            plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
            nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
            numero_carteirinha: str = Field(..., description="O número de identificação da carteirinha. Não deve conter espaços ou caracteres especiais.")

    Caso encontre a carteirinha, ou encontre informações relevantes, extraia as seguintes informações em formato JSON:
        {
        "confidence_its_carteirinha": "float",  # Valor entre 0.0 e 1.0 indicando a confiança
        "convenio": "string",
        "plano": "string",
        "nome_pessoa": "string",
        "numero_carteirinha": "string"
        }
    Note: Caso seja apenas uma pagina da web, pode ser que informaões relevantes estejam nessa pagina.
    Por exemplo, um convenio descrito é um forte indicio de que a informacao mais relevante pra nos, o numero da carteirinha, esteja presente.
    A presença de outros dados, como o nome do beneficiário ou o plano de saúde, também pode ser um indicativo útil, de que o numero da carteirinha esteja presente.

    Se nenhuma imagem contiver uma carteirinha de convênio de saúde, retorne:
    {
        "convenio": null,
        "plano": null,
        "nome_pessoa": null,
        "numero_carteirinha": null
    }

    

""",
"prompt_extracao_carteirinha_v3": """
        You are an expert in OCR and structured data extraction. Your job is to analyze the provided images and extract relevant information about health insurance cards.
        They are in portuguese so the next instructions are too.
        Você receberá uma ou mais imagens. Analise todas e verifique se alguma delas contém uma carteirinha de convênio de saúde.
        Uma carteirinha normalmente possui estes campos:
        - convenio: O nome da operadora do plano de saúde.
        - plano: O tipo ou nome do plano (ex: "Plano Prata", "Enfermaria").
        - nome_pessoa: O nome completo do beneficiário.
        - numero_carteirinha: O número de identificação ou matrícula da carteirinha. Não deve haver caracter especial, como  es
        onde, de forma mais tecnica:
            convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
            plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
            nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
            numero_carteirinha: str = Field(..., description="O número de identificação da carteirinha. Não deve conter espaços ou caracteres especiais.")
        
        Em muitos casos, as imagens não serão de carteirinhas, mas exibem informações relevantes. Ou tem a carteirinha no meio de uma pagina web. Caso encontre a carteirinha, ou encontre informações relevantes,
        extraia as seguintes informações em formato JSON:
        {
        "convenio": "string",
        "plano": "string",
        "nome_pessoa": "string",
        "numero_carteirinha": "string"
        }

        Os valores em cada campo nao devem conter caracteres especiais, definidos por r"[^a-zA-Z0-9\s]".
        Por exemplo, se um numero de carteirinha for "12345-6", ele deve ser armazenado como "123456".

        O  "convenio" dos convenios possiveis estao nessa lista, assim como a quantidade de digitos nos "numero_carteirinha".
        Voce deve utilizar essa lista para identificar o convenio e o numero de carteirinha extraidos.
        Por exemplo, uma imagem em que foram extraidos 3 numeros diferentes, e o nome do convenio, consulte a lista
        para entender qual numero de carteirinha correto. 
            [
                ("SUL AMERICA DIRETO BH", 17),
                ("STELLANTIS SAUDE MG", 17),
                ("SUL AMERICA", 20),
                ("CASSI", 16),
                ("CAIXA ECONOMICA FEDERAL", 16),
                ("BLUE COMPANY", 16),
                ("POSTAL SAUDE - CORREIOS", 16),
                ("IPSM", 16),
                ("UNIMED SEGUROS", 16),
                ("BRADESCO", 15),
                ("BRADESCO OPERADORA", 15),
                ("PLAN ASSISTE - MPF", 14),
                ("CARE PLUS", 12),
                ("PETROBRAS - REGAP", 12),
                ("VALE - AMS", 12),
                ("FUNDAFFEMG", 12),
                ("CEMIG SAUDE", 15),
                ("VALE - PASA", 10),
                ("AMIL", 9),
                ("AMIL VM (ANTIGA GOLDEN CROSS)", 9),
                ("COPASS", 8),
                ("SPA SAUDE", 5)
]
    Exemplo de retorno do campo convenio:
    "Caso um possivel nome de convenio encontrado possua BRADESCO saude brasil, o campo de convenio deve ser BRADESCO, conforme a lista de convenios descreve.
    Sempre consulte a lista."
    
    Caso o nome do convenio nao esteja na lista, retorne null.
    Caso o numero da carteirinha nao esteja de acordo com a quantidade de digitos esperada, os mais de um numero no conjunto de imagens
    onde nenhum for valido, retorne null.

    Os campos que não forem possiveis de serem identificados, deverao retornar null.

    Retorne apenas a sua resposta nesse formato:
        {
        "convenio": "string",
        "plano": "string",
        "nome_pessoa": "string",
        "numero_carteirinha": "string"
        }


"""
}

<>:57: SyntaxWarning: invalid escape sequence '\s'
<>:57: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2241/784351322.py:57: SyntaxWarning: invalid escape sequence '\s'
  "prompt_extracao_carteirinha_v3": """


In [77]:
# instantiate our agent

carteirinha_agent_prompt = prompts["prompt_extracao_carteirinha_v3"]

carteirinha_agent = AgentCarteirinha(model=bedrock_model, system_prompt=carteirinha_agent_prompt, data_model=CarteirinhaExtraida)

In [72]:
# send the image bytes to agent and get a response
response = carteirinha_agent.extract_text_from_pdf(pdf_bytes=example_blob)

```json
{
"convenio": "BRADESCO",
"plano": "SAUDE BRADESCO",
"nome_pessoa": "MARIA APARECIDA SILVA",
"numero_carteirinha": "123456789012345"
}
```

In [78]:
images_bytes_list

[b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x06\xf9\x00\x00\t\xde\x08\x00\x00\x00\x00\xa7\xa0vR\x00\x01\x00\x00IDATx\x9c\xec\xbdy\x94$\xd9]\xdf\xfb\x8b{c\x8f\x8c\x88\xdc\xb3\xaak\xe9\xee\xaa\xee\xd9g\xb4\x8d\xd0\x8a\x84\xc0\x18\x83\xb1\xb1\r\xc6\x07\xdb,\xf6\xb3y~\xd8`\xe0=\x83\r\xf616\x961f1\x9b\xd9m@\x180 l\tl\x16!\x19\t\xa1eF#i\xf6\xe9\x99\xe9\xb5\xbak\xc95"\x97\xd8\xe3\xdex\x7fD\xe4ZY]\xd9]]=3\xd4\xfd\x9e#\x9d\xe9O}\x7f7"\xee\xf2\xfbUd\xc6\x8d\xe2\x12`bbbbb:AB\xaf\xf4\t011111\xddU\xb1\xca\xc7\xc4\xc4\xc4\xc4t\xb2\xc4*\x1f\x13\x13\x13\x13\xd3\xc9\x12\xab|LLLLL\'K\xac\xf211111\x9d,\xb1\xca\xc7\xc4\xc4\xc4\xc4t\xb2\xc4*\x1f\x13\x13\x13\x13\xd3\xc9\x12\xab|LLLLL\'K\xac\xf211111\x9d,\xb1\xca\xc7\xc4\xc4\xc4\xc4t\xb2\xc4*\x1f\x13\x13\x13\x13\xd3\xc9\x12\xab|LLLLL\'K\xac\xf211111\x9d,\xb1\xca\xc7\xc4\xc4\xc4\xc4t\xb2\xc4*\x1f\x13\x13\x13\x13\xd3\xc9\x12\xab|LLLLL\'K\xac\xf211111\x9d,\xb1\xca\xc7\xc4\xc4\xc4\xc4t\xb2\xc4*\x1f\x13\x13\x13\x13\xd3\xc9\x12\xab|LLLLL\'K\xac\xf211111\x9d,\xb

In [79]:
# extract data from image bytes

response2 = carteirinha_agent.extract_data_from_images_bytes(list_images_bytes=images_bytes_list)

```json
{
"convenio": "BRADESCO",
"plano": "SAUDE NACIONAL FLEX E CA COPAR",
"nome_pessoa": "MARIA APARECIDA SILVA",
"numero_carteirinha": "123456789012345"
}
```

## funcionou com imagens e com pdf como blob. testar com as duas approachs. pdf párece ser muito melhor

In [57]:
# run for more 10 images and analize.